In [1]:
import os, argparse, typing
os.chdir("../../")

import numpy as np
import pandas as pd
import scanpy as sc
import torch 
from torch import nn
from torch import optim
from torch.utils.data import DataLoader
from torch.optim import lr_scheduler
from functools import partial
import pytorch_lightning as pl
from pytorch_lightning import callbacks 
from pytorch_lightning import loggers as pl_loggers

from PINN import models as models
from PINN import reader 
from PINN import functions as fns

In [2]:
import matplotlib.pyplot as plt
from PINN import pl
from importlib import reload

In [3]:
reload(reader)

<module 'PINN.reader' from '/ssd/users/Wergillius/Project/PINN_dynamics/PINN/reader.py'>

# load dataset

In [4]:
# load the anndata
ery_mk_ad = sc.read_h5ad("data/ery_mk.h5ad")

In [5]:
train_DS = reader.HigDimRe_AnnDS(AnnData=ery_mk_ad, n_dimension = 5, 
                               cellstate_key="DM_EigenVectors",  #'DM_EigenVector'
                                     nearby_cellstate=1, 
                                    collocation_points=300)


train_DS_t5 = reader.HigDimRe_AnnDS(AnnData=ery_mk_ad, n_timepoint=5, n_dimension = 5, 
                              cellstate_key="DM_EigenVectors",  #'DM_EigenVector'
                                     nearby_cellstate=1, 
                                    collocation_points=300)
T_b_max = np.log(train_DS_t5.popD['t'].max())

In [6]:
s_all = train_DS.s.float().requires_grad_()

t_b = train_DS.t_b.float().requires_grad_()

In [7]:
train_DS.popD['t']

array([  3,   7,  12,  27,  49,  76, 112, 161, 269])

In [39]:
def pred(Model, Train_DataSet, device= 'cuda:6'):

    relu_f = nn.ReLU()
    Model = Model.to(device)

    n_timepoint = len(Train_DataSet.popD['t'])
    
    s_all = Train_DataSet.s.float().requires_grad_()
    t_b = Train_DataSet.t_b.float().requires_grad_()

    u_pred_b = Model(s_all.to(device), t_b.to(device))
    # u_pred_b = relu_f(u_pred_b)
        
    
    u_pred_b = u_pred_b.detach().cpu().numpy().reshape(n_timepoint, -1)
    u_b = Train_DataSet.u_b.cpu().numpy().reshape(n_timepoint, -1)

    return u_b, u_pred_b

In [10]:
from Explore_Notebook.density import plotting_fn as pl_fn

In [11]:
# predict
def pred_vis(model_woPL, device):
    u, u_pred = pred(model_woPL, train_DS_t5, device)
    u_pred.shape
    t5_ad = train_DS_t5.adata
    for i, t in enumerate(train_DS_t5.popD['t']):
        t5_ad.obs[f'Day{t}_u'] = u[i]
        t5_ad.obs[f'Day{t}_woPL_upred'] = u_pred[i]
    u_pred.max(axis=1), u_pred.min(axis=1)
    pl_fn.umap_by_time(lambda x: f'Day{x}_u', t5_ad, train_DS_t5.popD['t'])
    pl_fn.umap_by_time(lambda x: f'Day{x}_woPL_upred', t5_ad,train_DS_t5.popD['t'])

## minibatched s and t

In [12]:
device = "cuda:5"

In [29]:
# s_v3, t_v3, and t_v3_norm

In [25]:
s_v3 = torch.tensor([[-6.6311e-03, -2.3923e-03,  8.7960e-04, -4.0171e-03, -4.9789e-03],
        [-6.6387e-03, -1.8259e-02, -6.0479e-03, -9.5295e-03,  4.8077e-03],
        [-6.6254e-03,  1.2532e-02,  1.2671e-02,  1.4230e-02,  7.1178e-03],
        [-6.6273e-03,  9.9622e-03,  6.1182e-03,  6.8527e-04,  5.2122e-03],
        [-6.6313e-03,  5.9351e-04, -3.6004e-03, -9.0528e-03, -5.5953e-03],
        [-6.6321e-03, -5.0750e-04, -5.4830e-03, -2.7304e-02,  2.1723e-03],
        [-6.6376e-03, -1.0235e-02, -1.9891e-02, -8.2493e-03, -1.1420e-02],
        [-6.6331e-03, -4.2069e-03, -7.3414e-03, -1.0625e-02, -1.1109e-02],
        [-6.6343e-03, -2.6453e-03, -1.3110e-02,  5.1000e-04, -8.8166e-03],
        [-6.6350e-03, -7.8187e-03, -9.5492e-04,  4.2350e-03,  1.3001e-02],
        [-6.6215e-03,  1.9327e-02,  2.5155e-02,  2.9324e-02,  1.7810e-02],
        [-6.6324e-03, -5.2045e-03, -2.6363e-03, -8.5030e-03, -8.3718e-03],
        [-6.6273e-03,  6.9299e-03,  9.2194e-03,  8.6709e-03,  2.7704e-03],
        [-6.6255e-03,  1.4797e-02,  1.0036e-02,  1.5627e-02,  7.4475e-03],
        [-6.6354e-03, -6.3893e-03, -1.5194e-02, -8.6450e-03, -1.3195e-02],
        [-6.6280e-03,  9.0629e-03,  4.5200e-03, -1.2778e-02,  1.1045e-02],
        [-6.6261e-03,  1.2203e-02,  9.5935e-03,  1.1904e-02,  5.2619e-03],
        [-6.6335e-03, -6.4123e-03, -6.7383e-03, -1.2849e-02, -1.1603e-02],
        [-6.6361e-03, -1.7169e-02,  5.7731e-03, -7.1571e-03, -8.3987e-03],
        [-6.6303e-03, -4.0070e-03,  1.9979e-02,  1.1699e-02,  1.4828e-03],
        [-6.6319e-03,  2.0680e-03, -7.1394e-03,  4.3469e-03, -5.8286e-03],
        [-6.6372e-03, -1.8535e-02,  2.2719e-03, -9.0948e-03, -3.9875e-03],
        [-6.6262e-03,  1.3327e-02,  8.9550e-03, -8.1368e-03,  1.6489e-02],
        [-6.6304e-03,  6.2020e-03, -4.8962e-03,  1.1792e-02, -4.0720e-03],
        [-6.6385e-03, -1.2510e-02, -2.3948e-02, -1.4921e-02, -2.1360e-02],
        [-6.6292e-03,  6.1745e-03,  1.5223e-03, -1.3056e-02,  5.3089e-03],
        [-6.6279e-03,  7.4082e-03,  6.0444e-03,  5.9134e-03,  1.1599e-03],
        [-6.6296e-03,  4.2018e-03,  6.7205e-04, -1.5545e-03, -2.5316e-03],
        [-6.6360e-03, -9.6027e-03, -4.5250e-03,  1.0488e-03,  9.1002e-03],
        [-6.6353e-03, -5.2826e-03, -1.1550e-02,  1.8157e-03,  9.3115e-04],
        [-6.6302e-03, -8.2828e-05,  2.9033e-03, -9.3564e-04, -3.0581e-03],
        [-6.6236e-03,  1.7399e-02,  1.6359e-02,  1.9176e-02,  1.1258e-02],
        [-6.6376e-03, -1.3171e-02, -8.4648e-03, -3.9908e-03,  7.0280e-03],
        [-6.6270e-03,  1.1511e-02,  7.0533e-03, -1.0296e-02,  1.4299e-02],
        [-6.6294e-03,  6.1079e-03, -1.5023e-04,  1.7263e-03, -1.8120e-03],
        [-6.6357e-03, -6.5757e-03, -1.4040e-02, -4.0998e-03, -6.5559e-03],
        [-6.6372e-03, -9.4565e-03, -2.0595e-02, -9.5944e-03, -1.6856e-02],
        [-6.6334e-03, -7.2926e-03,  9.2341e-03,  6.7727e-03,  1.3579e-02],
        [-6.6280e-03,  4.8104e-03,  8.4251e-03,  6.6330e-03,  1.9882e-03],
        [-6.6276e-03,  7.7234e-03,  7.1264e-03,  8.0495e-03,  1.5586e-03],
        [-6.6328e-03,  8.2116e-04, -6.7727e-03,  8.8076e-03,  3.7674e-03],
        [-6.6273e-03,  9.7421e-03,  6.2575e-03,  5.2007e-04,  5.0829e-03],
        [-6.6295e-03,  3.3409e-03,  2.4408e-03,  1.7764e-03, -2.6480e-03],
        [-6.6286e-03,  9.7531e-03,  3.2225e-04,  1.4918e-02,  5.7243e-04],
        [-6.6326e-03,  9.2871e-04, -9.3223e-03,  4.6646e-03, -6.2792e-03],
        [-6.6350e-03, -7.4450e-03, -1.2680e-02, -3.3090e-02, -7.7629e-03],
        [-6.6254e-03,  1.5307e-02,  1.0991e-02, -7.2653e-03,  2.0110e-02],
        [-6.6309e-03,  7.1688e-04, -1.9463e-03, -7.3078e-03, -5.0621e-03],
        [-6.6274e-03,  9.6053e-03,  6.1503e-03,  1.5022e-04,  5.2839e-03],
        [-6.6295e-03,  6.7519e-03, -8.1635e-04,  6.8277e-03, -1.8557e-03]],
       device=device, requires_grad=True)

t_v3 = torch.tensor([16., 13., 29., 19., 13.,  8., 13., 32., 33., 46., 28., 21., 14., 32.,
        27., 42., 35., 23., 43., 34., 19., 24., 35., 12., 15., 40., 16., 12.,
        29., 36., 13., 43., 47., 34.,  6.,  5., 27., 16., 28., 43., 40., 48.,
        48., 43., 34., 12., 30., 43., 17., 46.], device=device,
       requires_grad=True)

t_v3_norm = torch.log(t_v3) / T_b_max

In [ ]:
# s_v5, t_v5

In [15]:
s_v5 = torch.tensor([[-6.6282e-03,  5.1490e-03,  6.8462e-03,  5.6603e-03,  5.7692e-04],
        [-6.6296e-03,  5.1671e-03,  7.1462e-04, -1.4824e-02,  5.0426e-03],
        [-6.6292e-03,  4.5128e-03,  2.5087e-03,  8.7702e-04, -1.7431e-03],
        [-6.6353e-03, -1.2074e-02,  2.9166e-03, -8.4878e-04,  1.7508e-04],
        [-6.6300e-03,  3.7128e-03, -2.9950e-04,  7.6398e-04, -3.6299e-03],
        [-6.6297e-03,  4.1839e-03,  3.7354e-04, -2.0182e-03, -2.5023e-03],
        [-6.6271e-03,  5.8380e-03,  1.1932e-02,  1.0943e-02,  5.1947e-03],
        [-6.6297e-03,  4.9502e-03,  8.5241e-05, -1.3202e-02,  2.8582e-03],
        [-6.6274e-03,  5.6536e-03,  1.0385e-02,  9.1709e-03,  3.6441e-03],
        [-6.6296e-03,  5.3754e-03,  6.2529e-04, -1.8988e-02,  8.0959e-03],
        [-6.6325e-03,  9.8164e-04, -9.6588e-03,  3.0977e-03, -9.0720e-03],
        [-6.6303e-03,  3.7049e-03, -1.9973e-03, -3.3225e-03, -3.4764e-03],
        [-6.6299e-03,  4.4714e-03, -9.0601e-04, -9.3705e-03, -4.3445e-04],
        [-6.6283e-03,  5.0735e-03,  6.6192e-03,  5.9127e-03,  3.9136e-04],
        [-6.6283e-03,  5.1590e-03,  6.5312e-03,  4.6451e-03,  8.4177e-04],
        [-6.6355e-03, -8.2310e-03, -4.4669e-03,  2.6012e-03,  9.7422e-03],
        [-6.6270e-03,  5.8985e-03,  1.2435e-02,  1.1483e-02,  5.7516e-03],
        [-6.6294e-03,  4.3565e-03,  1.5291e-03, -6.3478e-04, -1.9630e-03],
        [-6.6297e-03,  5.0308e-03,  2.4927e-04, -1.3923e-02,  3.5333e-03],
        [-6.6297e-03,  4.9474e-03,  7.3376e-05, -1.3066e-02,  2.6143e-03],
        [-6.6294e-03,  4.3817e-03,  1.6446e-03, -1.9324e-04, -2.1327e-03],
        [-6.6283e-03,  5.0752e-03,  6.2971e-03,  4.9659e-03,  2.1892e-04],
        [-6.6291e-03,  4.5083e-03,  3.1970e-03,  2.8072e-03, -1.8063e-03],
        [-6.6334e-03, -4.1789e-04, -1.0918e-02,  4.2788e-03, -6.2806e-03],
        [-6.6296e-03,  5.2721e-03,  5.3660e-04, -1.7242e-02,  6.4376e-03],
        [-6.6294e-03,  4.2446e-03,  1.8290e-03,  1.7985e-03, -2.5591e-03],
        [-6.6280e-03,  5.2749e-03,  7.5516e-03,  6.1122e-03,  1.1664e-03],
        [-6.6288e-03,  4.7449e-03,  4.1184e-03,  2.8113e-03, -1.0072e-03],
        [-6.6281e-03,  5.1960e-03,  7.1770e-03,  6.0526e-03,  8.7391e-04],
        [-6.6296e-03,  5.0809e-03,  7.3787e-04, -1.3247e-02,  3.8306e-03],
        [-6.6278e-03,  5.4326e-03,  8.9208e-03,  7.8446e-03,  2.3547e-03],
        [-6.6329e-03,  3.4394e-04, -1.0100e-02,  3.6734e-03, -7.5252e-03],
        [-6.6275e-03,  5.6201e-03,  1.0233e-02,  9.2170e-03,  3.3015e-03],
        [-6.6296e-03,  5.3577e-03,  6.4616e-04, -1.8501e-02,  7.6893e-03],
        [-6.6297e-03,  5.0133e-03,  2.5283e-04, -1.3643e-02,  3.3687e-03],
        [-6.6295e-03,  5.1119e-03,  9.1843e-04, -1.3102e-02,  4.0275e-03],
        [-6.6293e-03,  4.7990e-03,  1.9781e-03, -4.5599e-03,  5.9477e-05],
        [-6.6296e-03,  5.2774e-03,  6.1625e-04, -1.7119e-02,  6.5387e-03],
        [-6.6306e-03,  3.1319e-03, -2.6660e-03, -7.0123e-04, -4.3983e-03],
        [-6.6291e-03,  4.8952e-03,  2.9958e-03, -2.6928e-03,  3.3597e-04],
        [-6.6296e-03,  5.3834e-03,  6.9667e-04, -1.8843e-02,  8.0964e-03],
        [-6.6295e-03,  4.2217e-03,  1.2961e-03,  4.2221e-04, -2.6069e-03],
        [-6.6297e-03,  4.9836e-03,  2.2472e-04, -1.3267e-02,  3.0027e-03],
        [-6.6296e-03,  5.3169e-03,  8.1126e-04, -1.7204e-02,  6.9929e-03],
        [-6.6272e-03,  5.7998e-03,  1.1595e-02,  1.0541e-02,  4.8359e-03],
        [-6.6348e-03, -1.4913e-02,  9.4141e-03, -4.0298e-03, -1.1419e-02],
        [-6.6298e-03,  4.5747e-03, -1.6516e-04, -8.2641e-03, -5.2426e-04],
        [-6.6287e-03,  5.1113e-03,  4.5686e-03, -8.6037e-04,  1.2382e-03],
        [-6.6298e-03,  4.7275e-03, -2.3802e-04, -1.0756e-02,  8.2585e-04],
        [-6.6297e-03,  5.1299e-03,  3.3029e-04, -1.5424e-02,  4.7847e-03]],
       device=device, requires_grad=True)

t_v5 = torch.tensor([[0.6385],
        [0.8469],
        [0.6385],
        [0.8469],
        [0.8469],
        [0.8469],
        [1.0000],
        [0.5000],
        [0.5000],
        [1.0000],
        [0.5000],
        [0.8469],
        [0.8469],
        [0.8469],
        [0.8469],
        [1.0000],
        [1.0000],
        [0.6385],
        [0.5000],
        [0.6385],
        [0.8469],
        [0.6385],
        [0.8469],
        [0.8469],
        [0.8469],
        [0.8469],
        [1.0000],
        [0.2823],
        [0.2823],
        [0.8469],
        [0.5000],
        [0.6385],
        [0.6385],
        [0.6385],
        [0.6385],
        [1.0000],
        [0.2823],
        [0.6385],
        [0.5000],
        [0.8469],
        [0.5000],
        [0.8469],
        [0.6385],
        [0.8469],
        [0.6385],
        [1.0000],
        [1.0000],
        [0.6385],
        [0.8469],
        [0.6385]], device=device, requires_grad=True)

# MLP PINN

In [20]:
# randomly inited 
u_theta = models.MLP_surrogate(channels = [6,32,32,1], activation_fn='Tanh')
model_rand = models.MLP_woD(u=u_theta, channels=[6,32])


# load model with pl method
model_v3 = models.MLP_woD.load_from_checkpoint("logs/ery_mk-DM_EigenVectors_multiBnch/MLP_woD/lightning_logs/version_3/checkpoints/epoch=1-total_loss=0.00007198.ckpt")
model_v3 = model_v3.to(device)

v5_ckpt_path = "logs/ery_mk-DM_EigenVectors_multiBnch/MLP_woD/lightning_logs/version_5/checkpoints/epoch=1-total_loss=0.00012448.ckpt"
model_v5 = models.MLP_woD.load_from_checkpoint(v5_ckpt_path)
model_v5 = model_v5.to(device)


# try another way to load the model
ckpt_v5 = torch.load(v5_ckpt_path)
u_theta2 = models.MLP_surrogate(channels = [6,32,32,1], activation_fn='Tanh')
model_init_v5 = models.MLP_woD(u=u_theta2, channels=[6,32])
model_init_v5.load_state_dict(ckpt_v5['state_dict'])

/ssd/users/Wergillius/miniforge3/envs/PINN_torch/lib/python3.9/site-packages/pytorch_lightning/utilities/parsing.py:208: Attribute 'u' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['u'])`.


<All keys matched successfully>

In [32]:
model_v5(s_v5.to(device), t_v5.to(device))

tensor([0.0015, 0.0008, 0.0011, 0.0029, 0.0013, 0.0011, 0.0026, 0.0008, 0.0022,
        0.0012, 0.0020, 0.0011, 0.0007, 0.0016, 0.0015, 0.0054, 0.0027, 0.0011,
        0.0008, 0.0006, 0.0012, 0.0014, 0.0014, 0.0027, 0.0007, 0.0013, 0.0021,
        0.0024, 0.0027, 0.0008, 0.0020, 0.0022, 0.0019, 0.0007, 0.0006, 0.0012,
        0.0019, 0.0006, 0.0015, 0.0011, 0.0009, 0.0012, 0.0006, 0.0008, 0.0021,
        0.0010, 0.0012, 0.0012, 0.0007, 0.0006], device='cuda:5',
       grad_fn=<SqueezeBackward1>)

In [36]:
model_v5(s_all[:50].to(device), t_v5.to(device))

tensor([ 0.0017,  0.0025,  0.0014,  0.0021,  0.0009,  0.0018,  0.0012,  0.0032,
         0.0053,  0.0018,  0.0021,  0.0017,  0.0015,  0.0055,  0.0010,  0.0014,
         0.0056,  0.0007,  0.0028,  0.0054,  0.0030,  0.0041,  0.0013,  0.0017,
         0.0023,  0.0040,  0.0057,  0.0024,  0.0037,  0.0001,  0.0010,  0.0021,
         0.0028,  0.0013,  0.0006, -0.0005,  0.0016,  0.0015,  0.0053,  0.0056,
         0.0010,  0.0014,  0.0017,  0.0014,  0.0021,  0.0027,  0.0040,  0.0016,
         0.0026,  0.0023], device='cuda:5', grad_fn=<SqueezeBackward1>)

In [37]:
model_v5(s_all[:50].to(device), t_b[:50].to(device))

tensor([0.0033, 0.0039, 0.0030, 0.0037, 0.0023, 0.0034, 0.0021, 0.0045, 0.0067,
        0.0028, 0.0034, 0.0032, 0.0030, 0.0071, 0.0025, 0.0023, 0.0069, 0.0022,
        0.0041, 0.0071, 0.0044, 0.0057, 0.0028, 0.0032, 0.0038, 0.0056, 0.0069,
        0.0029, 0.0042, 0.0016, 0.0023, 0.0037, 0.0045, 0.0029, 0.0021, 0.0005,
        0.0021, 0.0030, 0.0067, 0.0072, 0.0023, 0.0028, 0.0033, 0.0028, 0.0036,
        0.0036, 0.0052, 0.0031, 0.0040, 0.0038], device='cuda:5',
       grad_fn=<SqueezeBackward1>)

In [38]:
model_v5(s_all.to(device), t_b.to(device))

tensor([0.0033, 0.0039, 0.0030,  ..., 0.0018, 0.0035, 0.0012], device='cuda:5',
       grad_fn=<SqueezeBackward1>)

In [40]:
u, u_pred_b = pred(model_v5, train_DS_t5, device)

In [30]:
model_v3.u(s_v3.to(device),t_v3.to(device))

tensor([ 0.0374,  0.0277,  0.0368,  0.0417,  0.0273, -0.0017,  0.0271,  0.0330,
         0.0319,  0.0149,  0.0381,  0.0424,  0.0318,  0.0330,  0.0390,  0.0192,
         0.0289,  0.0419,  0.0186,  0.0305,  0.0419,  0.0416,  0.0283,  0.0228,
         0.0343,  0.0219,  0.0375,  0.0227,  0.0368,  0.0278,  0.0276,  0.0184,
         0.0137,  0.0297, -0.0121, -0.0145,  0.0391,  0.0379,  0.0379,  0.0185,
         0.0224,  0.0124,  0.0126,  0.0186,  0.0305,  0.0220,  0.0348,  0.0184,
         0.0393,  0.0149], device='cuda:5', grad_fn=<SqueezeBackward1>)

In [31]:
model_v3.u(s_v3.to(device),t_v3_norm.to(device))

tensor([ 1.7301e-03,  3.4955e-03,  5.5785e-04,  9.8116e-04,  1.2678e-03,
         2.3922e-03,  7.4583e-04,  1.1327e-03, -8.2649e-05,  2.0664e-03,
         5.6195e-04,  1.7857e-03,  1.0856e-03, -8.9258e-06,  5.3268e-04,
         1.6915e-03,  3.1164e-04,  1.6352e-03,  3.4654e-03,  2.8284e-03,
         3.7938e-05,  3.8111e-03,  1.6665e-03, -4.8568e-04,  5.9687e-04,
         1.5296e-03,  8.3502e-04,  9.7103e-04,  2.1469e-03,  6.0866e-04,
         1.5795e-03,  1.4073e-04,  2.1429e-03,  1.7265e-03,  4.3396e-04,
         7.1624e-04,  2.8087e-04,  3.1142e-03,  1.2170e-03,  5.4111e-04,
         8.3387e-05,  7.5451e-04,  6.9735e-04, -7.0255e-04, -2.5593e-04,
         2.5195e-03,  1.7724e-03,  1.0297e-03,  1.0628e-03, -1.7893e-04],
       device='cuda:5', grad_fn=<SqueezeBackward1>)